# Data Cleaning - Capa BRONZE

En este notebook se realiza la limpieza inicial de los datos para la capa BRONZE.

El objetivo es preparar la información para su posterior uso en el pipeline ETL.

In [1]:
import os

# Configurar variables de entorno para Hadoop en Windows
HADOOP_HOME = os.environ.get("HADOOP_HOME", "C:\\hadoop")

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["hadoop.home.dir"] = HADOOP_HOME
os.environ["PATH"] = f"{os.environ.get('PATH', '')};{HADOOP_HOME}\\bin"

In [2]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, trim, lower, regexp_replace
from pyspark.ml.feature import Imputer
import os

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [3]:
# Crear sesión de Spark

spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin_Bronze")
    .getOrCreate()
)

print("Spark inicializado")

Spark inicializado


## Cargar los conjuntos de datos

In [4]:
# Cargar datasets

INPUT_PATH = "../data/modified"
OUTPUT_PATH = "../data/processed/bronze"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Leer como texto para conservar identificadores y códigos postales
def cargar_csv(nombre):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", "UTF-8")
        .csv(f"{INPUT_PATH}/{nombre}_data.csv")
    )

users_spark = cargar_csv("users")
cards_spark = cargar_csv("cards")
transactions_spark = cargar_csv("transactions")

print("Datos cargados")

Datos cargados


## Cargar cuentas y obligaciones

Se incorporan las cuentas y las obligaciones de pago.

Los identificadores se conservan como texto y los importes se
convierten a decimal para mantener los centavos.

In [5]:
# Importar funciones adicionales

from pyspark.sql import functions as F


# Cargar datasets adicionales

accounts_spark = (
    spark.read
    .option("header", True)
    .csv(f"{INPUT_PATH}/accounts_data.csv")
)

obligations_spark = (
    spark.read
    .option("header", True)
    .csv(f"{INPUT_PATH}/obligations_payments_data.csv")
)


# Normalizar textos sin modificar identificadores

def preparar_textos(df):
    for campo in df.schema.fields:
        if campo.dataType.simpleString() == "string":
            valor = F.trim(F.col(campo.name))

            df = df.withColumn(
                campo.name,
                F.when(valor != "", valor)
            )

    return df


# Convertir importes

def convertir_importes(df, columnas):
    for columna in columnas:
        df = df.withColumn(
            columna,
            F.regexp_replace(
                F.col(columna).cast("string"),
                r"[$,]",
                ""
            )
        )

        df = df.withColumn(
            columna,
            F.expr(
                f"try_cast(`{columna}` as decimal(18, 2))"
            )
        )

    return df


# Preparar cuentas

accounts_spark = preparar_textos(accounts_spark)

accounts_spark = convertir_importes(
    accounts_spark,
    ["opening_balance", "closing_balance"]
)

accounts_spark = (
    accounts_spark
    .withColumn(
        "client_id",
        F.expr("try_cast(client_id as bigint)")
    )
    .withColumn(
        "overdraft_allowed",
        F.expr("try_cast(overdraft_allowed as boolean)")
    )
    .withColumn(
        "currency",
        F.upper("currency")
    )
)


# Preparar obligaciones

obligations_spark = preparar_textos(obligations_spark)

obligations_spark = convertir_importes(
    obligations_spark,
    [
        "contract_principal",
        "amount_due",
        "amount_paid",
        "outstanding_amount"
    ]
)

for columna in [
    "client_id",
    "installment_number",
    "payment_transaction_id",
    "days_past_due_as_of",
    "days_to_full_payment"
]:
    obligations_spark = obligations_spark.withColumn(
        columna,
        F.expr(f"try_cast(`{columna}` as bigint)")
    )

obligations_spark = (
    obligations_spark
    .withColumn(
        "annual_interest_rate",
        F.expr(
            "try_cast(annual_interest_rate as decimal(12, 6))"
        )
    )
    .withColumn(
        "currency",
        F.upper("currency")
    )
)

print("Cuentas y obligaciones cargadas")

Cuentas y obligaciones cargadas


## Explorar la información

In [6]:
# Mostrar primeras filas

users_spark.show(5)
cards_spark.show(5)
transactions_spark.show(5)

+----+-----------+--------------+----------+-----------+------+--------------------+---------+-----------+-----------------+-------------+----------+------------+----------------+--------------------+--------------------+----------------+----------------+-------+-----------+--------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address| latitude|  longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|           full_name|               email|            city|           state|country|postal_code|currency|
+----+-----------+--------------+----------+-----------+------+--------------------+---------+-----------+-----------------+-------------+----------+------------+----------------+--------------------+--------------------+----------------+----------------+-------+-----------+--------+
| 825|         53|            66|      1966|         11|Female|Calle Reforma 362...|19.705900|-101.194900|        527004.00|   1074528.00|2297034

In [7]:
# Mostrar esquema

users_spark.printSchema()
cards_spark.printSchema()
transactions_spark.printSchema()

root
 |-- id: string (nullable = true)
 |-- current_age: string (nullable = true)
 |-- retirement_age: string (nullable = true)
 |-- birth_year: string (nullable = true)
 |-- birth_month: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- num_credit_cards: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- currency: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- 

## Identificar valores faltantes

In [8]:
# Contar valores nulos en cada dataset

users_nulls = users_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_spark.columns
])

cards_nulls = cards_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_spark.columns
])

transactions_nulls = transactions_spark.select([
    count(
        when(col(c).isNull(), c),
    ).alias(c)
    for c in transactions_spark.columns
])

print("Valores faltantes en users:")
users_nulls.show()

print("Valores faltantes en cards:")
cards_nulls.show()

print("Valores faltantes en transactions:")
transactions_nulls.show()

Valores faltantes en users:
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+---------+-----+----+-----+-------+-----------+--------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|full_name|email|city|state|country|postal_code|currency|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+---------+-----+----+-----+-------+-----------+--------+
|  0|         86|            65|        63|         55|    84|      0|       0|        0|               64|           46|        63|          48|              62|        0|    0|   0|    0|      0|          0|       0|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-------

## Limpieza de datos para capa BRONZE

In [9]:
# Limpiar strings
def clean_string(df, column):
    return df.withColumn(
        column,
        F.lower(F.trim(F.col(column)))
    )


# Limpiar importes conservando los centavos
def clean_currency(df, column):
    return convertir_importes(df, [column])


# Imputar con moda
def impute_mode(df, column):
    fila = (
        df.filter(F.col(column).isNotNull())
        .groupBy(column)
        .count()
        .orderBy(F.desc("count"), F.asc(column))
        .first()
    )

    return df.fillna({column: fila[column]}) if fila else df


# Imputar con mediana
def impute_median(df, column):
    valores = df.approxQuantile(column, [0.5], 0.001)

    return df.fillna({column: valores[0]}) if valores else df


# Convertir tipos sin detener la limpieza por valores inválidos
def convertir_tipos(df, columnas, tipo):
    for columna in columnas:
        df = df.withColumn(
            columna,
            F.expr(f"try_cast(`{columna}` as {tipo})")
        )

    return df

In [10]:
# Limpiar users_spark

users_spark = preparar_textos(users_spark)

users_spark = convertir_tipos(
    users_spark,
    [
        "id",
        "retirement_age",
        "birth_year",
        "birth_month",
        "credit_score",
        "num_credit_cards"
    ],
    "bigint"
)

users_spark = convertir_tipos(
    users_spark,
    ["current_age", "latitude", "longitude"],
    "double"
)

for column in [
    "per_capita_income",
    "yearly_income",
    "total_debt"
]:
    users_spark = clean_currency(users_spark, column)


# Conservar nombres y ubicaciones con su escritura original
for column in [
    "full_name",
    "address",
    "city",
    "state",
    "country"
]:
    users_spark = users_spark.withColumn(
        column,
        F.regexp_replace(
            F.trim(F.col(column)),
            r"\s+",
            " "
        )
    )

users_spark = clean_string(users_spark, "email")
users_spark = clean_string(users_spark, "gender")

users_spark = users_spark.withColumn(
    "currency",
    F.upper("currency")
)


# Validar rangos e imputar los campos previstos
users_spark = (
    users_spark
    .withColumn(
        "current_age",
        F.when(F.col("current_age") < 18, 18)
        .when(F.col("current_age") > 120, 120)
        .otherwise(F.col("current_age"))
    )
    .withColumn(
        "credit_score",
        F.when(F.col("credit_score") < 300, 300)
        .when(F.col("credit_score") > 850, 850)
        .otherwise(F.col("credit_score"))
    )
)

users_spark = impute_median(users_spark, "current_age")
users_spark = impute_mode(users_spark, "gender")

print("Limpieza de users completada")

Limpieza de users completada


In [11]:
# Limpiar cards_spark

cards_spark = preparar_textos(cards_spark)

cards_spark = convertir_tipos(
    cards_spark,
    ["id", "client_id", "year_pin_last_changed"],
    "bigint"
)

cards_spark = convertir_tipos(
    cards_spark,
    ["num_cards_issued"],
    "double"
)

for column in [
    "card_brand",
    "card_type",
    "has_chip",
    "card_on_dark_web"
]:
    cards_spark = clean_string(cards_spark, column)

cards_spark = clean_currency(cards_spark, "credit_limit")

cards_spark = cards_spark.withColumn(
    "currency",
    F.upper("currency")
)

cards_spark = impute_median(cards_spark, "num_cards_issued")
cards_spark = impute_mode(cards_spark, "card_brand")

cards_spark = cards_spark.withColumn(
    "credit_limit",
    F.when(F.col("credit_limit") < 0, 0)
    .otherwise(F.col("credit_limit"))
)

# Los límites faltantes se estiman en SILVER con el modelo existente
print("Limpieza de cards completada")

Limpieza de cards completada


## Limpiar los movimientos

Se normalizan los textos y los importes.

No se imputan montos faltantes ni se recortan los importes,
porque estas operaciones modificarían los saldos.

La convención de cada registro se conserva para separar las
transacciones anteriores de los movimientos de cuenta.

In [12]:
# Limpiar transactions_spark

transactions_spark = preparar_textos(
    transactions_spark
)


# Normalizar categorías

string_columns = [
    "use_chip",
    "merchant_city",
    "merchant_state",
    "errors",
    "movement_type",
    "category",
    "direction",
    "income_source",
    "amount_convention"
]

for column in string_columns:
    transactions_spark = transactions_spark.withColumn(
        column,
        F.lower(F.col(column))
    )


# Convertir importes sin imputar ni recortar

transactions_spark = convertir_importes(
    transactions_spark,
    [
        "amount",
        "balance_before",
        "balance_after"
    ]
)


# Convertir identificadores numéricos

for column in ["id", "client_id", "card_id", "merchant_id", "mcc"]:
    transactions_spark = transactions_spark.withColumn(
        column,
        F.expr(f"try_cast(`{column}` as bigint)")
    )


# Preparar moneda y recurrencia

transactions_spark = (
    transactions_spark
    .withColumn(
        "currency",
        F.upper("currency")
    )
    .withColumn(
        "is_recurring",
        F.expr("try_cast(is_recurring as boolean)")
    )
)

print("Limpieza de transactions completada")

Limpieza de transactions completada


## Verificar los cambios

In [13]:
# Verificar valores nulos despues de la limpieza

users_nulls_clean = users_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_spark.columns
])

cards_nulls_clean = cards_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_spark.columns
])

transactions_nulls_clean = transactions_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in transactions_spark.columns
])

print("Valores faltantes en users despues de limpieza:")
users_nulls_clean.show()

print("Valores faltantes en cards despues de limpieza:")
cards_nulls_clean.show()

print("Valores faltantes en transactions despues de limpieza:")
transactions_nulls_clean.show()

Valores faltantes en users despues de limpieza:
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+---------+-----+----+-----+-------+-----------+--------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|full_name|email|city|state|country|postal_code|currency|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+---------+-----+----+-----+-------+-----------+--------+
|  0|          0|            65|        63|         55|     0|      0|       0|        0|               64|           46|        63|          48|              62|        0|    0|   0|    0|      0|          0|       0|
+---+-----------+--------------+----------+-----------+------+-------+------

In [14]:
# Mostrar muestra de datos limpios

print("Muestra de users limpios:")
users_spark.show(5)

print("Muestra de cards limpios:")
cards_spark.show(5)

print("Muestra de transactions limpios:")
transactions_spark.show(5)

Muestra de users limpios:
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+--------------------+--------------------+----------------+----------------+-------+-----------+--------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|           full_name|               email|            city|           state|country|postal_code|currency|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+--------------------+--------------------+----------------+----------------+-------+-----------+--------+
| 825|       53.0|            66|      1966|         11|female|Calle Reforma 362...| 19.7059|-101.1949|        527004.00|   1074

## Guardar datos en capa BRONZE

In [15]:
# Guardar datos limpios en formato Parquet

datasets = {
    "users": users_spark,
    "cards": cards_spark,
    "transactions": transactions_spark,
    "accounts": accounts_spark,
    "obligations_payments": obligations_spark
}

for nombre, datos in datasets.items():
    datos.write.mode("overwrite").parquet(
        f"{OUTPUT_PATH}/{nombre}.parquet"
    )

print("Datos guardados en capa BRONZE")

Datos guardados en capa BRONZE


## Finalizar sesión de Spark

In [16]:
# Finalizar Spark
spark.stop()
print("Pipeline completado")

Pipeline completado
